## Iteration Changelog
- Updated EDA to read from `data/cleanedGambling.csv` and include stack + board-risk feature exploration.
- Added focused summaries for card strength, SPR dynamics, and shared-board risk patterns.
- Open issues: add temporal drift slices by month/tournament for production monitoring.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = ROOT / "data" / "cleanedGambling.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(df.shape)
df.head(2)

## Dataset Overview

### Key Questions
- How do new stack-aware features distribute?
- How does shared-board risk vary by outcome?
- Do card-strength signals separate win/loss groups?

In [ ]:
eda_cols = [
    "preflop_preflop_strength", "flop_hand_strength", "turn_hand_strength", "river_hand_strength",
    "preflop_hero_stack_bb", "flop_spr", "turn_spr", "river_spr",
    "flop_board_shared_strength_risk", "turn_board_shared_strength_risk", "river_board_shared_strength_risk",
    "won_flag"
]
eda_cols = [c for c in eda_cols if c in df.columns]

df[eda_cols].describe().T

In [ ]:
# Stack distributions and SPR behavior
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(["preflop_hero_stack_bb", "flop_spr", "turn_spr"]):
    if col in df.columns:
        df[col].hist(bins=40, ax=ax[i])
        ax[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Shared-board risk by outcome
risk_col = "river_board_shared_strength_risk"
if risk_col in df.columns:
    summary = df.groupby("won_flag")[risk_col].agg(["mean", "median", "std", "count"])
    print(summary)
    ax = df.boxplot(column=risk_col, by="won_flag", figsize=(6, 4))
    plt.title("River Shared-Board Risk by Win Flag")
    plt.suptitle("")
    plt.show()

## Key Findings
- Stack pressure features are now available across streets and can be analyzed jointly with card strength.
- Shared-board risk introduces a clearer caution signal on dangerous board textures.
- Card-strength and board-risk features provide complementary information rather than purely redundant signals.

## Next Steps
- Slice by tournament/date for drift checks.
- Compare these EDA trends against model feature importances over retrains.

## EDA Summary & Key Insights

### Dataset Overview
This notebook performs descriptive analysis on the cleaned dataset generated by preprocessing.

### Analysis Focus
- Stack pressure (`hero_stack_bb`, `effective_stack_bb`, `spr`)
- Card strength progression by street
- Shared-board risk (`board_shared_strength_risk`) and outcome relationship